In [1]:
import os
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, precision_recall_curve
from xgboost import XGBClassifier
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

pd.options.mode.chained_assignment = None  # default='warn'

# Config the matlotlib backend as plotting inline in IPython
%matplotlib inline

In [2]:
# data = pd.read_csv('banking.csv', header = 0)
# data.to_hdf('banking.h5', key='data', mode='w') 
data = pd.read_hdf('banking.h5', 'data')
print(f"DR: {data['y'].mean():.6f}")
print(data.shape)
display(data)

DR: 0.112654
(41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,44,blue-collar,married,basic.4y,unknown,yes,no,cellular,aug,thu,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,0
1,53,technician,married,unknown,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.021,5195.8,0
2,28,management,single,university.degree,no,yes,no,cellular,jun,thu,...,3,6,2,success,-1.7,94.055,-39.8,0.729,4991.6,1
3,39,services,married,high.school,no,no,no,cellular,apr,fri,...,2,999,0,nonexistent,-1.8,93.075,-47.1,1.405,5099.1,0
4,55,retired,married,basic.4y,no,yes,no,cellular,aug,fri,...,1,3,1,success,-2.9,92.201,-31.4,0.869,5076.2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,59,retired,married,high.school,unknown,no,yes,telephone,jun,thu,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.866,5228.1,0
41184,31,housemaid,married,basic.4y,unknown,no,no,telephone,may,thu,...,2,999,0,nonexistent,1.1,93.994,-36.4,4.860,5191.0,0
41185,42,admin.,single,university.degree,unknown,yes,yes,telephone,may,wed,...,3,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
41186,48,technician,married,professional.course,no,no,yes,telephone,oct,tue,...,2,999,0,nonexistent,-3.4,92.431,-26.9,0.742,5017.5,0


In [3]:
train_data, test_data = train_test_split(data, test_size = 0.3, stratify = data.y, random_state=42)
print(train_data.shape)
print(test_data.shape)

(28831, 21)
(12357, 21)


In [4]:
target = 'y'
all_columns = train_data.columns

string_cols = train_data.select_dtypes(include='object').columns.tolist()
numeric_cols = train_data.select_dtypes(include='number').columns.tolist()

all_feats = [x for x in all_columns if ((x in numeric_cols + string_cols) and (x != target))]

df_feats = pd.DataFrame({
    'col_name':  all_feats
})

df_feats = df_feats.assign(main_type = '')
df_feats = df_feats.assign(gini_train = 0.0)
df_feats = df_feats.assign(gini_test = 0.0)
df_feats = df_feats.assign(delta_gini = 0.0)
df_feats = df_feats.assign(precision_train = 0.0)
df_feats = df_feats.assign(precision_test = 0.0)
df_feats = df_feats.assign(recall_train = 0.0)
df_feats = df_feats.assign(recall_test = 0.0)

# ~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.

for i in range(len(df_feats)):
    feat = df_feats.iloc[i, df_feats.columns.get_loc('col_name')]
    
    if feat in string_cols:
        X_train = train_data[[feat]].copy()
        X_train[feat] = X_train[feat].astype('category')
        y_train = train_data[target]
        
        X_test = test_data[[feat]].copy()
        X_test[feat] = X_test[feat].astype('category')
        y_test = test_data[target]

        df_feats.iloc[i, df_feats.columns.get_loc('main_type')] = 'string'

    elif feat in numeric_cols:
        X_train = train_data[[feat]].copy()
        X_train[feat] = pd.to_numeric(X_train[feat], errors="coerce")
        y_train = train_data[target]
        
        X_test = test_data[[feat]].copy()
        X_test[feat] = pd.to_numeric(X_test[feat], errors="coerce")
        y_test = test_data[target]

        df_feats.iloc[i, df_feats.columns.get_loc('main_type')] = 'numeric'
    else:
        break

    model = XGBClassifier(
        booster = 'gbtree',
        objective = "binary:logistic",  # important for probability output
        enable_categorical = True, 
        tree_method = 'hist',
        eval_metric = "auc",
        n_jobs = os.cpu_count(),
        random_state = 42,
        verbosity = 0
    )

    model.fit(X_train, y_train)

    proba_train = model.predict_proba(X_train)
    proba_test = model.predict_proba(X_test)

    X_train['p_0'] = proba_train[:, 0]
    X_train['p_1'] = proba_train[:, 1]
    X_train['target'] = y_train.values

    auc_train = roc_auc_score(X_train['target'], X_train['p_1'])

    precision, recall, thresholds = precision_recall_curve(X_train['target'], X_train['p_1'])
    f1 = np.where(
        (precision[:-1] + recall[:-1]) > 0,
        2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1]),
        0
    )
    best_thresh = thresholds[np.argmax(f1)]

    # Convert prob → binary prediction
    X_train['target_pred'] = np.where(X_train['p_1'] > best_thresh, 1, 0)
    
    precision_train = precision_score(X_train['target'], X_train['target_pred'], zero_division = 0)
    recall_train = recall_score(X_train['target'], X_train['target_pred'], zero_division = 0)

    X_test['p_0'] = proba_test[:, 0]
    X_test['p_1'] = proba_test[:, 1]
    X_test['target']  = y_test.values

    auc_test = roc_auc_score(X_test['target'], X_test['p_1'])

    # Convert prob → binary prediction
    X_test['target_pred']  = np.where(X_test['p_1']  > best_thresh, 1, 0)
    
    precision_test = precision_score(X_test['target'], X_test['target_pred'], zero_division = 0)
    recall_test = recall_score(X_test['target'], X_test['target_pred'], zero_division = 0)

    gini_train = 2.0 * auc_train - 1.0
    gini_test = 2.0 * auc_test - 1.0
    delta_gini = np.absolute(gini_train - gini_test)

    df_feats.iloc[i, df_feats.columns.get_loc('gini_train')] = gini_train
    df_feats.iloc[i, df_feats.columns.get_loc('gini_test')] = gini_test
    df_feats.iloc[i, df_feats.columns.get_loc('delta_gini')] = delta_gini
    df_feats.iloc[i, df_feats.columns.get_loc('precision_train')] = precision_train
    df_feats.iloc[i, df_feats.columns.get_loc('precision_test')] = precision_test
    df_feats.iloc[i, df_feats.columns.get_loc('recall_train')] = recall_train
    df_feats.iloc[i, df_feats.columns.get_loc('recall_test')] = recall_test

# ~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.~.
    
df_feats = df_feats.sort_values(by = 'gini_test', ascending = False) 
display(df_feats.reset_index(drop=True))

,col_name,main_type,gini_train,gini_test,delta_gini,precision_train,precision_test,recall_train,recall_test
0,duration,numeric,0.646955,0.644039,0.002916,0.418740,0.430686,0.466441,0.459770
1,euribor3m,numeric,0.584752,0.572191,0.012561,0.438259,0.444103,0.533251,0.519397
2,cons_price_idx,numeric,0.573942,0.559790,0.014152,0.468052,0.457923,0.464594,0.433908
3,cons_conf_idx,numeric,0.573942,0.559790,0.014152,0.468052,0.457923,0.464594,0.433908
4,nr_employed,numeric,0.545997,0.537187,0.008811,0.490646,0.485774,0.355296,0.331178
5,emp_var_rate,numeric,0.526244,0.519851,0.006393,0.470619,0.468504,0.273707,0.256466
6,month,string,0.310515,0.328206,0.017691,0.456189,0.484429,0.201970,0.201149
7,contact,string,0.219052,0.223804,0.004752,0.000000,0.000000,0.000000,0.000000
8,job,string,0.219152,0.223171,0.004019,0.156025,0.159836,0.474754,0.476293
9,poutcome,string,0.231661,0.218411,0.013250,0.000000,0.000000,0.000000,0.000000
